In [ ]:
include("main_utils.jl")
include("data_setup.jl")
include("comix_uk_time_series.jl")
include("vis_utils.jl")
include("mglm_utils.jl")
include("danon_utils.jl")

default_plot_setting()

## 1. Load Danon 2013 data

Reshape `dt_Leon_Danon_2013/` into the project's contact-table schema
(`:part_id_d`, `:date`, `:cnt_home`, `:duration_multi`, `:c_number`).
Danon's duration codes `{-1, 0, 1, 2, 3}` are mapped to project levels
`{"NA", 1, 2, 3, 4}` (a K=4 scale with bin midpoints 5 / 20 / 45 min
and a `>60 min` open-ended top bin). Home stratification uses
`C_Wheres_1 == 1`. The 43 contact rows with `C_Number ∈ {-1, 0}` are
dropped on load.

**Group-contact disaggregation.** Each raw row with `C_Number = k`
represents a *group* contact with `k` individuals; the loader now
expands these into `k` rows so that one row = one individual contact
(`disaggregate = true`, default). This is what makes
`nrow(df) ≈ Σ P_total_contacts` and lets `contact_degrees(...)`
return the per-person degree directly. `c_number` is preserved on each
disaggregated row so the original group size remains queryable.

**Duration caveat for groups.** Danon's duration field is ambiguous
when `C_Number > 1`: some respondents recorded the *time spent with the
entire group* (i.e. the group event duration) rather than the per-
individual contact time. Under the standard co-presence model these
coincide — the responder is exposed to each group member for the full
event. We adopt that interpretation as default (`duration_mode = :as_is`,
keep the recorded duration on every disaggregated copy). The alternative
reading — that recorded time should be split across group members —
is supported via `duration_mode = :per_person` and exercised in §6 as
a sensitivity.

In [ ]:
df_raw, df_part = read_danon_contacts(; disaggregate = false)
df,     _       = read_danon_contacts()   # disaggregate = true, :as_is

println("# raw contact rows (after C_Number >= 1 filter): ", nrow(df_raw))
println("# group-contact rows (C_Number > 1):             ", sum(df_raw.c_number .> 1),
        "  (", round(100 * mean(df_raw.c_number .> 1); digits = 1), "%)")
println("# rows after disaggregation:                     ", nrow(df))
println("# responders (df_part):                          ", nrow(df_part))
println("home / non-home contacts (post-disagg): ",
    sum(df.cnt_home .== "true"), " / ", sum(df.cnt_home .== "false"))
println("NA-duration contacts (post-disagg):     ", sum(df.duration_multi .== "NA"))

cn_labels = ["1", "2", "3", "4-5", "6-10", "11-50", "50+"]
cn_bin = map(df_raw.c_number) do k
    k == 1 ? cn_labels[1] :
    k == 2 ? cn_labels[2] :
    k == 3 ? cn_labels[3] :
    k <= 5 ? cn_labels[4] :
    k <= 10 ? cn_labels[5] :
    k <= 50 ? cn_labels[6] : cn_labels[7]
end
println("\nC_Number distribution (raw rows):")
for lab in cn_labels
    n = sum(cn_bin .== lab)
    @printf("  %-6s  %6d rows  (%5.2f%%)\n", lab, n, 100 * n / length(cn_bin))
end

deg_rows = combine(groupby(df, :part_id_d), nrow => :n_rows)
chk = innerjoin(deg_rows, @select(df_part, :part_id_d, :p_total_contacts),
                on = :part_id_d)
chk_ok = @subset(chk, .!ismissing.(:p_total_contacts))
println("\nDisaggregated rows vs. P_total_contacts:")
println("  N responders cross-checked: ", nrow(chk_ok))
println("  correlation: ",
    round(cor(chk_ok.n_rows, chk_ok.p_total_contacts); digits = 3),
    "    median ratio: ",
    round(median(chk_ok.n_rows ./ chk_ok.p_total_contacts); digits = 3))
first(chk_ok, 5)

## 2. Missing duration pattern over degree

Per (responder × setting): degree `n` (count of contact rows) and `n_na`
(count whose duration is `"NA"`). The scatter shows the missing
fraction `n_na/n` against `n` on log-x, with marker size proportional
to `log10(#responders)` via the same `_marker_size_log` / `_xticks_for_max`
helpers as the 2j proportion panels.

In [ ]:
function _danon_miss_by_setting(df, df_part; setting::Symbol)
    sub = setting === :home    ? @subset(df, :cnt_home .== "true")  :
          setting === :nonhome ? @subset(df, :cnt_home .== "false") :
          setting === :all     ? df :
          error("setting must be :all, :home, or :nonhome")

    grp = combine(groupby(sub, [:part_id_d, :date])) do s
        (; n = nrow(s),
            n_na = sum(_is_dur_na.(s[:, :duration_multi])))
    end
    keys_part = unique(@select(df_part, :part_id_d, :date))
    grp = leftjoin(keys_part, grp, on = [:part_id_d, :date])
    grp.n    = coalesce.(grp.n,    0)
    grp.n_na = coalesce.(grp.n_na, 0)
    return @subset(grp, :n .> 0)
end

function _plot_miss_panel(df, df_part, setting::Symbol; title)
    grp = _danon_miss_by_setting(df, df_part; setting = setting)
    grp = @transform(grp, :p_na = :n_na ./ :n)
    pts = combine(groupby(grp, [:n, :p_na]), nrow => :count)
    max_n = maximum(pts.n)
    return scatter(pts.n, pts.p_na;
        ms = _marker_size_log(pts.count; scale = 2.0),
        msw = 0, alpha = 0.55,
        xlabel = "degree", ylabel = "fraction NA",
        xscale = :log10, xticks = _xticks_for_max(max_n),
        xlim = (0.9, max_n * 1.1), ylim = (-0.02, 1.02),
        legend = false, title = title)
end

p_miss_h = _plot_miss_panel(df, df_part, :home;    title = "missing duration — home")
p_miss_n = _plot_miss_panel(df, df_part, :nonhome; title = "missing duration — non-home")
plot(p_miss_h, p_miss_n; layout = (1, 2), size = (900, 400))

### 2.1 Duration proportion by degree — split by participant group-contact status

Same drop-NA-keep-`n` pipeline as §3, but partitioning **participants** into
those whose every contact has `C_Number == 1` (no group contacts) and those
with at least one group contact (`C_Number > 1`). The split is at the
responder level, so each `(part_id_d, date)` cell appears in exactly one
panel column.

In [ ]:
part_class = combine(groupby(df_raw, :part_id_d),
                     :c_number => (x -> any(x .> 1)) => :has_group)
ids_solo  = Set(@subset(part_class, .!:has_group).part_id_d)
ids_group = Set(@subset(part_class,  :has_group).part_id_d)
println("# participants with only c_number==1 contacts: ", length(ids_solo))
println("# participants with ≥1 c_number>1 contact:     ", length(ids_group))

df_solo  = @subset(df, in.(:part_id_d, Ref(ids_solo)))
df_group = @subset(df, in.(:part_id_d, Ref(ids_group)))

function _emp_props_for(df_sub; setting)
    inp = prepare_dm_inputs(df_sub; setting = setting,
                            outcome = :duration_multi,
                            K = 4, dropna_keep_n = true)
    return _emp_props_by_n_with_denom(inp.Y, inp.n, inp.n_obs, 4)
end

p_solo_h  = _props_panel_base(_emp_props_for(df_solo;  setting = "home"),
                              4, category_names_viz[:duration_danon];
                              title = "solo only — home")
p_solo_n  = _props_panel_base(_emp_props_for(df_solo;  setting = "non-home"),
                              4, category_names_viz[:duration_danon];
                              title = "solo only — non-home")
p_group_h = _props_panel_base(_emp_props_for(df_group; setting = "home"),
                              4, category_names_viz[:duration_danon];
                              title = "has group — home")
p_group_n = _props_panel_base(_emp_props_for(df_group; setting = "non-home"),
                              4, category_names_viz[:duration_danon];
                              title = "has group — non-home")

plot(p_solo_h, p_solo_n, p_group_h, p_group_n;
     layout = (2, 2), size = (1000, 800))

### 2.2 Total contact duration over degree (continuous, midpoint-based)

Collapse `duration_multi ∈ {1,2,3,4}` to its bin midpoint
(`_DANON_T_MID_FINITE = (5.0, 20.0, 45.0, 60.0)` minutes), sum within
each responder, and scatter the per-person total against the per-person
degree — same solo / has-group split as §2.1, additionally split by
home / non-home setting.

NA-duration contacts are kept in the degree count (so the x-axis
matches §2.1's `dropna_keep_n` convention) but excluded from the
duration sum. Because `df` is in `:as_is` mode, each member of a group
contact inherits the original group's recorded duration, so the
y-axis is **person-minutes of contact** — the same quantity that §5's
weighting normalises by `d_max`.

In [ ]:
# Continuous duration per contact = bin midpoint (NA → missing, dropped from sum)
function _dur_minutes(d)
    _is_dur_na(d) && return missing
    return _DANON_T_MID_FINITE[parse(Int, d)]
end

function _person_totals(df_sub; setting::Symbol)
    sub = setting === :home    ? @subset(df_sub, :cnt_home .== "true")  :
          setting === :nonhome ? @subset(df_sub, :cnt_home .== "false") :
          setting === :all     ? df_sub :
          error("setting must be :all, :home, or :nonhome")
    sub = @transform(sub, :dur_min = _dur_minutes.(:duration_multi))
    return combine(groupby(sub, :part_id_d)) do s
        (; degree    = nrow(s),
           total_dur = sum(skipmissing(s.dur_min)))
    end
end

function _scatter_total_dur(pp; title, color)
    pts = @subset(pp, :total_dur .> 0)        # log10 needs > 0
    max_n = maximum(pts.degree)
    return scatter(pts.degree, pts.total_dur;
        ms = 3, msw = 0, alpha = 0.45, color = color,
        xlabel = "degree",
        ylabel = "total contact duration (min)",
        xscale = :log10, yscale = :log10,
        xticks = _xticks_for_max(max_n),
        xlim   = (0.9, max_n * 1.1),
        legend = false, title = title)
end

c_solo, c_group = palette(:default)[1], palette(:default)[2]

p22_solo_h  = _scatter_total_dur(_person_totals(df_solo;  setting = :home);
                                 title = "solo only — home",     color = c_solo)
p22_solo_n  = _scatter_total_dur(_person_totals(df_solo;  setting = :nonhome);
                                 title = "solo only — non-home", color = c_solo)
p22_group_h = _scatter_total_dur(_person_totals(df_group; setting = :home);
                                 title = "has group — home",     color = c_group)
p22_group_n = _scatter_total_dur(_person_totals(df_group; setting = :nonhome);
                                 title = "has group — non-home", color = c_group)

plot(p22_solo_h, p22_solo_n, p22_group_h, p22_group_n;
     layout = (2, 2), size = (1000, 800))

## 3. Proportion of duration categories over degree

Empirical category proportions per degree-bin, using the
drop-NA-keep-`n` variant from 2j: the bin x-axis is the full
per-cell degree (NA contacts included), and proportions are computed
over only the non-NA contacts.

In [ ]:
inp_home = prepare_dm_inputs(df; setting = "home",     outcome = :duration_multi,
                                 K = 4, dropna_keep_n = true)
inp_non  = prepare_dm_inputs(df; setting = "non-home", outcome = :duration_multi,
                                 K = 4, dropna_keep_n = true)

emp_home = _emp_props_by_n_with_denom(inp_home.Y, inp_home.n, inp_home.n_obs, 4)
emp_non  = _emp_props_by_n_with_denom(inp_non.Y,  inp_non.n,  inp_non.n_obs,  4)

p_prop_h = _props_panel_base(emp_home, 4, category_names_viz[:duration_danon];
                              title = "duration proportion — home")
p_prop_n = _props_panel_base(emp_non,  4, category_names_viz[:duration_danon];
                              title = "duration proportion — non-home")
plot(p_prop_h, p_prop_n; layout = (1, 2), size = (900, 400))

## 4. Dirichlet-multinomial regression (MGLM)

Fit `Y ~ log(degree)` per setting with `fit_mglm_dm`; compare AIC/BIC
against the intercept-only null. Overlays predicted μ with a 95%
Gaussian band (analytical BetaBinomial variance) on the §3 empirical
proportions.

In [ ]:
fit_home = fit_mglm_dm(inp_home.X, inp_home.Y)
fit_non  = fit_mglm_dm(inp_non.X,  inp_non.Y)

null_home = fit_mglm_dm(inp_home.X, inp_home.Y; intercept_only = true)
null_non  = fit_mglm_dm(inp_non.X,  inp_non.Y;  intercept_only = true)

println("home vs null:     ", mglm_compare(fit_home, null_home))
println("non-home vs null: ", mglm_compare(fit_non,  null_non))
println()
mglm_dm_show(fit_home); println()
mglm_dm_show(fit_non)

In [ ]:
function _overlay_dm_panel(emp, fit, K, names; title)
    p = _props_panel_base(emp, K, names; title = title)
    n_grid = collect(1:maximum(emp.n))
    pred = mglm_dm_predict(fit, n_grid)
    for k in 1:K
        plot!(p, n_grid, pred.μ[:, k];
              ribbon = (pred.μ[:, k] .- pred.lower[:, k],
                        pred.upper[:, k] .- pred.μ[:, k]),
              color = palette(:default)[k], lw = 1.5, label = "")
    end
    return p
end

function _overlay_dm_panel_one_cat(emp, fit, k, names; title)
    max_n  = maximum(emp.n)
    n_grid = collect(1:max_n)
    pred   = mglm_dm_predict(fit, n_grid)
    p = plot(xlabel = "degree", ylabel = "proportion",
             title = title, legend = :topright,
             ylim = (-0.02, 1.02), xscale = :log10,
             xticks = _xticks_for_max(max_n), xlim = (0.9, max_n * 1.1))
    scatter!(p, emp.n, emp[!, Symbol("p$k")];
             ms = _marker_size_log(emp.nobs), msw = 0, alpha = 0.7,
             color = palette(:default)[k], label = names[k])
    plot!(p, n_grid, pred.μ[:, k];
          ribbon = (pred.μ[:, k] .- pred.lower[:, k],
                    pred.upper[:, k] .- pred.μ[:, k]),
          color = palette(:default)[k], lw = 1.5, label = "")
    return p
end

names_dur = category_names_viz[:duration_danon]
panels_h  = [_overlay_dm_panel_one_cat(emp_home, fit_home, k, names_dur;
                title = string("DM fit — ", names_dur[k], " (home)"))     for k in 1:4]
panels_n  = [_overlay_dm_panel_one_cat(emp_non,  fit_non,  k, names_dur;
                title = string("DM fit — ", names_dur[k], " (non-home)")) for k in 1:4]
plot(panels_h..., panels_n...; layout = (2, 4), size = (1600, 700))

## 5. Effective contact-degree CCDF

Three weighting variants per setting (`all` / `home` / `non-home`),
each overlaying unweighted, weighted (NA → <10 min, level 1), and
weighted with DM-imputed NA. Weight rule from `inst/4_Danon_analysis.md`:
`w = 1` for `>60 min`, otherwise `w = t_mid / d_max` with `d_max = 60 min`.

In [ ]:
d_max = 60
print_duration_weights_danon(d_max)

deg_all  = contact_degrees(df, df_part; setting = :all)
deg_home = contact_degrees(df, df_part; setting = :home)
deg_non  = contact_degrees(df, df_part; setting = :nonhome)

wdeg_all  = contact_degrees_danon(df, df_part; setting = :all,
                                  weighted = true, d_max = d_max)
wdeg_home = contact_degrees_danon(df, df_part; setting = :home,
                                  weighted = true, d_max = d_max)
wdeg_non  = contact_degrees_danon(df, df_part; setting = :nonhome,
                                  weighted = true, d_max = d_max)

wdeg_home_imp = contact_degrees_dm_imputed_danon(df, df_part, fit_home;
                                                 setting = :home,    d_max = d_max)
wdeg_non_imp  = contact_degrees_dm_imputed_danon(df, df_part, fit_non;
                                                 setting = :nonhome, d_max = d_max)
wdeg_all_imp  = wdeg_home_imp .+ wdeg_non_imp;

In [ ]:
plot_weighting_compare(deg_all, wdeg_all, wdeg_all_imp;
                       setting_label = "all")

In [ ]:
plot_weighting_compare(deg_home, wdeg_home, wdeg_home_imp;
                       setting_label = "home")

In [ ]:
plot_weighting_compare(deg_non, wdeg_non, wdeg_non_imp;
                       setting_label = "non-home")

## 6. Sensitivity: per-person duration reinterpretation

The disaggregation in §1 propagates the recorded group duration to each
disaggregated copy (`duration_mode = :as_is`, co-presence). To probe
sensitivity to the alternative reading — that for `C_Number = k > 1` the
respondent reported the *total group event time* and the per-individual
contact time is `t_mid / k` — we reload with `duration_mode = :per_person`,
refit the DM, and overlay the weighted CCDF against §5.

Because the rebinning pushes group contacts toward shorter bins, the
sensitivity primarily lowers the weighted degree in **non-home** (where
group contacts concentrate); the home weighted degree barely moves.

In [ ]:
df_pp, _ = read_danon_contacts(; duration_mode = :per_person)

# Compare duration distribution shift on group contacts.
function _dur_freq(x)
    levels = ["NA", "1", "2", "3", "4"]
    return [count(==(l), x) for l in levels]
end
group_mask     = df.c_number    .> 1
group_mask_pp  = df_pp.c_number .> 1
println("Duration distribution on disaggregated group rows (C_Number > 1):")
println("  level           NA       1       2       3       4")
@printf("  :as_is      %7d %7d %7d %7d %7d\n",
        _dur_freq(df.duration_multi[group_mask])...)
@printf("  :per_person %7d %7d %7d %7d %7d\n",
        _dur_freq(df_pp.duration_multi[group_mask_pp])...)

In [ ]:
inp_home_pp = prepare_dm_inputs(df_pp; setting = "home",     outcome = :duration_multi,
                                       K = 4, dropna_keep_n = true)
inp_non_pp  = prepare_dm_inputs(df_pp; setting = "non-home", outcome = :duration_multi,
                                       K = 4, dropna_keep_n = true)

fit_home_pp = fit_mglm_dm(inp_home_pp.X, inp_home_pp.Y)
fit_non_pp  = fit_mglm_dm(inp_non_pp.X,  inp_non_pp.Y)

emp_home_pp = _emp_props_by_n_with_denom(inp_home_pp.Y, inp_home_pp.n, inp_home_pp.n_obs, 4)
emp_non_pp  = _emp_props_by_n_with_denom(inp_non_pp.Y,  inp_non_pp.n,  inp_non_pp.n_obs,  4)

p_fit_h_pp = _overlay_dm_panel(emp_home_pp, fit_home_pp, 4,
              category_names_viz[:duration_danon];
              title = "DM fit — home (:per_person)")
p_fit_n_pp = _overlay_dm_panel(emp_non_pp,  fit_non_pp, 4,
              category_names_viz[:duration_danon];
              title = "DM fit — non-home (:per_person)")
plot(p_fit_h_pp, p_fit_n_pp; layout = (1, 2), size = (900, 400))

In [ ]:
wdeg_home_pp = contact_degrees_danon(df_pp, df_part; setting = :home,
                                     weighted = true, d_max = d_max)
wdeg_non_pp  = contact_degrees_danon(df_pp, df_part; setting = :nonhome,
                                     weighted = true, d_max = d_max)
wdeg_all_pp  = contact_degrees_danon(df_pp, df_part; setting = :all,
                                     weighted = true, d_max = d_max)

function _summ(label, deg)
    @printf("  %-22s  mean = %7.2f  median = %5.1f  q95 = %6.1f\n",
            label, mean(deg), median(deg), quantile(deg, 0.95))
end
println("Weighted degree summaries (NA → level 1):")
println("  -- home --")
_summ("§5  :as_is",      wdeg_home)
_summ("§6  :per_person", wdeg_home_pp)
println("  -- non-home --")
_summ("§5  :as_is",      wdeg_non)
_summ("§6  :per_person", wdeg_non_pp)
println("  -- all --")
_summ("§5  :as_is",      wdeg_all)
_summ("§6  :per_person", wdeg_all_pp)

In [ ]:
function _ccdf_sensitivity_panel(x_unw, x_w_asis, x_w_pp; setting_label)
    p = plot(; xaxis = :log10, xlim = (0.1, 10_000), size = (700, 450),
              title = "CCDF — $setting_label  (duration_mode sensitivity)")
    plot_ccdf_continuous!(p, x_unw;     label = "unweighted",              color = :black)
    plot_ccdf_continuous!(p, x_w_asis;  label = "weighted, :as_is",        color = :orange)
    plot_ccdf_continuous!(p, x_w_pp;    label = "weighted, :per_person",   color = :purple)
    return p
end

plot(_ccdf_sensitivity_panel(deg_home, wdeg_home, wdeg_home_pp; setting_label = "home"),
     _ccdf_sensitivity_panel(deg_non,  wdeg_non,  wdeg_non_pp;  setting_label = "non-home"),
     _ccdf_sensitivity_panel(deg_all,  wdeg_all,  wdeg_all_pp;  setting_label = "all");
     layout = (1, 3), size = (1500, 450))